## Step 1: Data Loading

In [ ]:
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
import torchvision.transforms as T
import torch

# 一步构建图像变换
img_proc = T.Compose([T.Resize(224), T.ToTensor(), 
                      T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])

# 直接加载并配置
img_data = ImageFolder('dataset/', transform=img_proc)
img_loader = DataLoader(img_data, batch_size=16, shuffle=False, pin_memory=True)


## Step2: Feature Extraction

In [ ]:
# Database feature extraction and storage utility
import numpy as np
import pg8000.native
from rich.progress import track

# Initialize database connection
db = pg8000.native.Connection(
    user="postgres",
    password="postgres",
    host="localhost",
    database="facedb",
    port=5432
)

# Setup feature table schema
db.run("""
CREATE TABLE IF NOT EXISTS image_features (
    id SERIAL PRIMARY KEY,
    batch_id SMALLINT,
    image_id SMALLINT,
    channel_id SMALLINT,
    y_pos SMALLINT,
    x_pos SMALLINT,
    pixel_value REAL
)
""")

def store_image_data(image_loader, sampling_rate=0.1):
    """Store sampled image pixels to database."""
    stride = int(np.ceil(1.0/sampling_rate))
    feature_count = 0
    threshold = 1e-3  # Ignore near-zero values
    
  
    for idx, batch in enumerate(track(image_loader, description="Processing batches")):
        x_batch, _ = batch
        img_array = x_batch.detach().numpy()
    
        records = []
        b_size, ch_count, h, w = img_array.shape
        
        for b in range(b_size):
            for ch in range(ch_count):
                for i in range(0, h, stride):
                    for j in range(0, w, stride):
                        # Get pixel value with rounding
                        val = np.round(float(img_array[b, ch, i, j]), 4)
                        # Skip values close to zero
                        if abs(val) > threshold:
                            records.append([idx, b, ch, i, j, val])
        
        # Batch insert records if any exist
        if records:
            params = []
            for r in records:
                params.append(tuple(r))
                
            db.run("""
                INSERT INTO image_features 
                (batch_id, image_id, channel_id, y_pos, x_pos, pixel_value)
                VALUES %s
            """, params)
            
            feature_count += len(records)
            print(f"Batch {idx}: added {len(records)} features")
    
    print(f"⭐ Extraction complete! Stored {feature_count} features")
    return feature_count

features_stored = store_image_data(dl, 0.1)
db.close()


## Step 3: Model Profiling

In [ ]:
import onnx
import time
import numpy as np
import onnxruntime as ort

class ModelProfiler:
    def __init__(self, model_path):
        self.model = onnx.load(model_path)
        self.session = ort.InferenceSession(model_path)
        self.input_name = self.session.get_inputs()[0].name
    
    def profile(self, runs=50):
        # Create random input
        test_input = np.random.randn(1, 3, 224, 224).astype(np.float32)
        # Benchmark
        t0 = time.time()
        for _ in range(runs):
            self.session.run(None, {self.input_name: test_input})
        avg_ms = (time.time() - t0) / runs * 1000
        return len(self.model.graph.initializer), avg_ms

profiler = ModelProfiler("resnext50.onnx")
params, latency = profiler.profile()
print(f"Parameters: {params}, Time: {latency:.2f}ms")


## Step 4: Data Preparation

In [ ]:
import onnx
import time
import numpy as np
import psycopg2
from psycopg2.extras import execute_batch

class ModelDBTransformer:
    """Transform ONNX model parameters into database tables"""
    
    def __init__(self, connection_string):
        # Database connection setup
        self.connection = psycopg2.connect(connection_string)
        self.cursor = self.connection.cursor()
        self.cleanup_list = []
        
    def __del__(self):
        # Auto-cleanup
        if hasattr(self, 'connection') and self.connection:
            if hasattr(self, 'cursor') and self.cursor:
                self.cursor.close()
            self.connection.close()
    
    def execute_sql(self, statement, params=None, batch=False, data=None):
        """Execute SQL with proper error handling"""
        try:
            if batch and data:
                execute_batch(self.cursor, statement, data)
            elif params:
                self.cursor.execute(statement, params)
            else:
                self.cursor.execute(statement)
            self.connection.commit()
        except Exception as e:
            self.connection.rollback()
            print(f"SQL Error: {e}")
            raise
    
    def analyze_model(self, model_file):
        """Load model and extract stats"""
        network = onnx.load(model_file)
        print(f"Network Structure:")
        print(f"• Total nodes: {len(network.graph.node)}")
        print(f"• Weight tensors: {len(network.graph.initializer)}")
        return network
    
    def extract_weight_tables(self, network):
        """Convert model weights to database-friendly format"""
        tensor_table_map = {}
        
        for weight in network.graph.initializer:
            # Parse tensor info
            weight_id = weight.name
            dimensions = tuple(dim for dim in weight.dims)
            
            # Extract numeric data
            tensor_data = None
            if weight.data_type == 1:  # FLOAT
                tensor_data = np.frombuffer(weight.raw_data, dtype=np.float32).reshape(dimensions)
            else:
                continue  # Skip non-float tensors
                
            # Create table name (sanitized)
            table_id = f"wt_{weight_id.replace('.', '_').lower()}"
            tensor_table_map[table_id] = [tensor_data]
            
            # Output info for significant tensors
            if len(dimensions) > 1:
                print(f"✓ Weight: {weight_id} → Table: {table_id} ({dimensions})")
                
        return tensor_table_map
    
    def import_weights_to_db(self, weight_tables):
        """Store weight tensors in database tables"""
        create_statements = []
        insert_operations = []
        drop_statements = []
        param_records = {}
        
        # Process each weight tensor
        for table_name, tensor_list in weight_tables.items():
            tensor = tensor_list[0]
            shape = tensor.shape
            
            # Handle different tensor dimensions
            if len(shape) == 2:  # 2D weights (fc layers, etc)
                create_sql = f"CREATE TABLE IF NOT EXISTS {table_name} (output_unit SMALLINT, input_unit SMALLINT, weight_value REAL)"
                insert_sql = f"INSERT INTO {table_name} (output_unit, input_unit, weight_value) VALUES (%s, %s, %s)"
                
                # Flatten the tensor for insertion
                records = []
                for i in range(shape[0]):
                    for j in range(shape[1]):
                        records.append((i, j, float(tensor[i, j])))
                        
            else:  # 3D/4D weights (conv layers, etc)
                create_sql = f"CREATE TABLE IF NOT EXISTS {table_name} (filter_group SMALLINT, filter_id SMALLINT, element_id SMALLINT, weight_value REAL)" 
                insert_sql = f"INSERT INTO {table_name} (filter_group, filter_id, element_id, weight_value) VALUES (%s, %s, %s, %s)"
                
                # Flatten the multi-dimensional tensor
                records = []
                for g in range(shape[0]):
                    for f in range(shape[1]):
                        for e in range(shape[2]):
                            records.append((g, f, e, float(tensor[g, f, e])))
            
            # Store for batch execution
            param_records[table_name] = records
            create_statements.append(create_sql)
            insert_operations.append((insert_sql, table_name))
            drop_statements.append(f"DROP TABLE IF EXISTS {table_name};")
        
        # Execute all create statements
        for stmt in create_statements:
            self.execute_sql(stmt)
        
        # Execute all insert operations
        for op in insert_operations:
            self.execute_sql(op[0], batch=True, data=param_records[op[1]])
            
        # Save drop statements for cleanup
        with open("model_cleanup.sql", 'w') as f:
            f.write('\n'.join(drop_statements))
    
    def create_conv_mapping(self, name, input_size, kernel, stride, pad, channels):
        """Create mapping tables for convolution operations"""
        if name in ["existing_mapping1", "existing_mapping2"]:
            return  # Skip existing mappings
            
        h, w = input_size, input_size
        create_sql = f"""
        CREATE TABLE IF NOT EXISTS {name} (
            channel_id SMALLINT, 
            input_pos SMALLINT, 
            output_pos SMALLINT, 
            kernel_pos SMALLINT
        )
        """
        self.execute_sql(create_sql)
        
        # Generate mapping data
        mappings = []
        output_idx = 0
        
        for y_out in range(0, h + 2*pad - kernel + 1, stride):
            for x_out in range(0, w + 2*pad - kernel + 1, stride):
                kernel_idx = 0
                for c in range(channels):
                    for ky in range(kernel):
                        for kx in range(kernel):
                            y_in = y_out + ky
                            x_in = x_out + kx
                            
                            if y_in >= pad and y_in < h+pad and x_in >= pad and x_in < w+pad:
                                input_idx = (y_in-pad) * w + (x_in-pad)
                                mappings.append((c, input_idx, output_idx, kernel_idx))
                            kernel_idx += 1
                output_idx += 1
        
        # Batch insert the mappings
        insert_sql = f"INSERT INTO {name} (channel_id, input_pos, output_pos, kernel_pos) VALUES (%s, %s, %s, %s)"
        self.execute_sql(insert_sql, batch=True, data=mappings)
        self.cleanup_list.append(name)
    
    def create_grouped_conv_mapping(self, name, input_size, kernel, stride, pad, channels, groups):
        """Create mapping tables for grouped convolution operations"""
        skip_list = ["skip_map1", "skip_map2", "skip_map3", "skip_map4", 
                     "skip_map5", "skip_map6", "skip_map7", "skip_map8", "skip_map9"]
        if name in skip_list:
            return
            
        h = w = input_size
        
        # Create mapping table
        create_sql = f"""
        CREATE TABLE IF NOT EXISTS {name} (
            group_id SMALLINT,
            channel_id SMALLINT, 
            input_pos SMALLINT, 
            output_pos SMALLINT, 
            kernel_pos SMALLINT
        )
        """
        self.execute_sql(create_sql)
        
        # Calculate channel group assignments
        channels_per_group = channels // groups
        channel_groups = [c // channels_per_group for c in range(channels)]
        
        # Generate mapping data
        mappings = []
        output_idx = 0
        kernel_positions_per_group = kernel * kernel * (channels // groups)
        
        for y_out in range(0, h + 2*pad - kernel + 1, stride):
            for x_out in range(0, w + 2*pad - kernel + 1, stride):
                pos = 0
                for c in range(channels):
                    for ky in range(kernel):
                        for kx in range(kernel):
                            y_in = y_out + ky
                            x_in = x_out + kx
                            
                            if y_in >= pad and y_in < h+pad and x_in >= pad and x_in < w+pad:
                                input_idx = (y_in-pad) * w + (x_in-pad)
                                group_id = channel_groups[c]
                                mappings.append((
                                    group_id, 
                                    c, 
                                    input_idx, 
                                    output_idx, 
                                    pos % kernel_positions_per_group
                                ))
                            pos += 1
                output_idx += 1
                
        # Batch insert the mappings
        insert_sql = f"""
        INSERT INTO {name} 
        (group_id, channel_id, input_pos, output_pos, kernel_pos) 
        VALUES (%s, %s, %s, %s, %s)
        """
        self.execute_sql(insert_sql, batch=True, data=mappings)
        self.cleanup_list.append(name)
    
    def transform_model(self, model_path):
        """Main function to transform ONNX model to DB tables"""
        start = time.time()
        
        # Load and analyze model
        print(f"Processing model: {model_path}")
        network = self.analyze_model(model_path)
        
        # Extract weights as tables
        weight_tables = self.extract_weight_tables(network)
        print(f"Extracted {len(weight_tables)} weight tables")
        
        # Import weights to database 
        self.import_weights_to_db(weight_tables)
        
        # Create conv layer mappings
        self.create_conv_mapping("input_conv_map", 224, 7, 2, 3, 3)
        self.create_conv_mapping("block1_conv_map", 56, 3, 1, 1, 64)
        
       
connection_str = "dbname=facedb user=postgres password=postgres host=localhost"
transformer = ModelDBTransformer(connection_str)
transformer.transform_model("resnetx50.onnx")


## Step 5: Query Composition

In [ ]:
WITH
Conv201_fwd0 AS (
    select batch_id, K.kernel_id, F.matrix_id as tuple_id,
    sum(F.value*K.weight_value) as value
        FROM input_feature_map F INNER JOIN wt_conv201_weight K
        on F.order_id=K.element_id
    GROUP BY F.batch_id, K.kernel_id, F.matrix_id
),
Relu203_fwd0 AS (
    select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
        FROM Conv201_fwd0 F
),
Relu203_fwd0_mapped AS (
    select batch_id, F.kernel_id, matrix_id, order_id, value
        FROM Relu203_fwd0 F INNER JOIN input_conv_map M
        on F.kernel_id = M.channel_id and F.tuple_id = M.input_pos
),
MaxPool205_fwd0 AS (
    select batch_id, kernel_id, matrix_id as tuple_id, max(value) as value
        FROM Relu203_fwd0_mapped
    GROUP BY batch_id, kernel_id, matrix_id
),
Conv206_fwd0 AS (
    select batch_id, K.filter_id, F.tuple_id as tuple_id,
    sum(F.value*K.weight_value) as value
        FROM MaxPool205_fwd0 F INNER JOIN wt_conv206_weight K
        on F.kernel_id=K.element_id
    GROUP BY F.batch_id, K.filter_id, F.tuple_id
),
Conv207_fwd0 AS (
    select batch_id, K.filter_id, F.tuple_id as tuple_id,
    sum(F.value*K.weight_value) as value
        FROM MaxPool205_fwd0 F INNER JOIN wt_conv207_weight K
        on F.kernel_id=K.element_id
    GROUP BY F.batch_id, K.filter_id, F.tuple_id
),
Relu208_fwd0 AS (
    select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
        FROM Conv206_fwd0 F
),
Relu208_fwd0_mapped AS (
    select batch_id, group_id, output_pos, kernel_pos, value
        FROM Relu208_fwd0 F INNER JOIN block1_stage1_map M
        on F.kernel_id = M.channel_id and F.tuple_id = M.input_pos
),
Conv209_fwd0 AS (
    select batch_id, (F.group_id*4+filter_id) as kernel_id, F.output_pos as tuple_id,
    sum(F.value*K.weight_value) as value
        FROM Relu208_fwd0_mapped F INNER JOIN wt_conv209_weight K
        on F.kernel_pos=K.element_id and F.group_id=K.filter_group
    GROUP BY F.batch_id, F.group_id, K.filter_id, F.output_pos
),
Relu210_fwd0 AS (
    select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
        FROM Conv209_fwd0 F
),
Conv211_fwd0 AS (
    select batch_id, K.filter_id, F.tuple_id as tuple_id,
    sum(F.value*K.weight_value) as value
        FROM Relu210_fwd0 F INNER JOIN wt_conv211_weight K
        on F.kernel_id=K.element_id
    GROUP BY F.batch_id, K.filter_id, F.tuple_id
),
Add212_fwd0 AS (
    select A.batch_id, A.kernel_id, A.tuple_id,
    A.value + B.value as value
        FROM Conv207_fwd0 A INNER JOIN Conv211_fwd0 B
        on A.batch_id=B.batch_id and A.kernel_id=B.kernel_id
        and A.tuple_id=B.tuple_id
),
Relu213_fwd0 AS (
    select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
        FROM Add212_fwd0 F
),
Conv214_fwd0 AS (
    select batch_id, K.filter_id, F.tuple_id as tuple_id,
    sum(F.value*K.weight_value) as value
        FROM Relu213_fwd0 F INNER JOIN wt_conv214_weight K
        on F.kernel_id=K.element_id
    GROUP BY F.batch_id, K.filter_id, F.tuple_id
),
Relu215_fwd0 AS (
    select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
        FROM Conv214_fwd0 F
),
Relu215_fwd0_mapped AS (
    select batch_id, group_id, output_pos, kernel_pos, value
        FROM Relu215_fwd0 F INNER JOIN block1_stage2_map M
        on F.kernel_id = M.channel_id and F.tuple_id = M.input_pos
),
Conv216_fwd0 AS (
    select batch_id, (F.group_id*4+filter_id) as kernel_id, F.output_pos as tuple_id,
    sum(F.value*K.weight_value) as value
        FROM Relu215_fwd0_mapped F INNER JOIN wt_conv216_weight K
        on F.kernel_pos=K.element_id and F.group_id=K.filter_group
    GROUP BY F.batch_id, F.group_id, K.filter_id, F.output_pos
),
Relu217_fwd0 AS (
    select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
        FROM Conv216_fwd0 F
),
Conv218_fwd0 AS (
    select batch_id, K.filter_id, F.tuple_id as tuple_id,
    sum(F.value*K.weight_value) as value
        FROM Relu217_fwd0 F INNER JOIN wt_conv218_weight K
        on F.kernel_id=K.element_id
    GROUP BY F.batch_id, K.filter_id, F.tuple_id
),
Add219_fwd0 AS (
    select A.batch_id, A.kernel_id, A.tuple_id,
    A.value + B.value as value
        FROM Relu213_fwd0 A INNER JOIN Conv218_fwd0 B
        on A.batch_id=B.batch_id and A.kernel_id=B.kernel_id
        and A.tuple_id=B.tuple_id
),
Relu220_fwd0 AS (
    select batch_id, kernel_id, tuple_id, GREATEST(value,.0) as value
        FROM Add219_fwd0 F
),
Conv221_fwd0 AS (
    select batch_id, K.filter_id, F.tuple_id as tuple_id,
    sum(F.value*K.weight_value) as value
        FROM Relu220_fwd0 F INNER JOIN wt_conv221_weight K
        on F.kernel_id=K.element_id
    GROUP BY F.batch_id, K.filter_id, F.tuple_id
),
Relu222_fwd0 AS (
    select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
        FROM Conv221_fwd0 F
),
Relu222_fwd0_mapped AS (
    select batch_id, group_id, output_pos, kernel_pos, value
        FROM Relu222_fwd0 F INNER JOIN block1_stage3_map M
        on F.kernel_id = M.channel_id and F.tuple_id = M.input_pos
),
Conv223_fwd0 AS (
    select batch_id, (F.group_id*4+filter_id) as kernel_id, F.output_pos as tuple_id,
    sum(F.value*K.weight_value) as value
        FROM Relu222_fwd0_mapped F INNER JOIN wt_conv223_weight K
        on F.kernel_pos=K.element_id and F.group_id=K.filter_group
    GROUP BY F.batch_id, F.group_id, K.filter_id, F.output_pos
),
Relu224_fwd0 AS (
    select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
        FROM Conv223_fwd0 F
),
Conv225_fwd0 AS (
    select batch_id, K.filter_id, F.tuple_id as tuple_id,
    sum(F.value*K.weight_value) as value
        FROM Relu224_fwd0 F INNER JOIN wt_conv225_weight K
        on F.kernel_id=K.element_id
    GROUP BY F.batch_id, K.filter_id, F.tuple_id
),
Add226_fwd0 AS (
    select A.batch_id, A.kernel_id, A.tuple_id,
    A.value + B.value as value
        FROM Relu220_fwd0 A INNER JOIN Conv225_fwd0 B
        on A.batch_id=B.batch_id and A.kernel_id=B.kernel_id
        and A.tuple_id=B.tuple_id
),
Relu227_fwd0 AS (
    select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
        FROM Add226_fwd0 F
),
Conv228_fwd0 AS (
    select batch_id, K.filter_id, F.tuple_id as tuple_id,
    sum(F.value*K.weight_value) as value
        FROM Relu227_fwd0 F INNER JOIN wt_conv228_weight K
        on F.kernel_id=K.element_id
    GROUP BY F.batch_id, K.filter_id, F.tuple_id
),
Relu229_fwd0 AS (
    select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
        FROM Conv228_fwd0 F
),
Relu229_fwd0_mapped AS (
    select batch_id, group_id, output_pos, kernel_pos, value
        FROM Relu229_fwd0 F INNER JOIN block1_stage4_map M
        on F.kernel_id = M.channel_id and F.tuple_id = M.input_pos
),
Conv230_fwd0 AS (
    select batch_id, (F.group_id*4+filter_id) as kernel_id, F.output_pos as tuple_id,
    sum(F.value*K.weight_value) as value
        FROM Relu229_fwd0_mapped F INNER JOIN wt_conv230_weight K
        on F.kernel_pos=K.element_id and F.group_id=K.filter_group
    GROUP BY F.batch_id, F.group_id, K.filter_id, F.output_pos
),
Relu231_fwd0 AS (
    select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
        FROM Conv230_fwd0 F
),
Conv232_fwd0 AS (
    select batch_id, K.filter_id, F.tuple_id as tuple_id,
    sum(F.value*K.weight_value) as value
        FROM Relu231_fwd0 F INNER JOIN wt_conv232_weight K
        on F.kernel_id=K.element_id
    GROUP BY F.batch_id, K.filter_id, F.tuple_id
),
Add233_fwd0 AS (
    select A.batch_id, A.kernel_id, A.tuple_id,
    A.value + B.value as value
        FROM Relu227_fwd0 A INNER JOIN Conv232_fwd0 B
        on A.batch_id=B.batch_id and A.kernel_id=B.kernel_id
        and A.tuple_id=B.tuple_id
),
Relu234_fwd0 AS (
    select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
        FROM Add233_fwd0 F
),
Conv235_fwd0 AS (
    select batch_id, K.filter_id, F.tuple_id as tuple_id,
    sum(F.value*K.weight_value) as value
        FROM Relu234_fwd0 F INNER JOIN wt_conv235_weight K
        on F.kernel_id=K.element_id
    GROUP BY F.batch_id, K.filter_id, F.tuple_id
),
Relu236_fwd0 AS (
    select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
        FROM Conv235_fwd0 F
),
Relu236_fwd0_mapped AS (
    select batch_id, group_id, output_pos, kernel_pos, value
        FROM Relu236_fwd0 F INNER JOIN block2_stage1_map M
        on F.kernel_id = M.channel_id and F.tuple_id = M.input_pos
),
Conv237_fwd0 AS (
    select batch_id, (F.group_id*4+filter_id) as kernel_id, F.output_pos as tuple_id,
    sum(F.value*K.weight_value) as value
        FROM Relu236_fwd0_mapped F INNER JOIN wt_conv237_weight K
        on F.kernel_pos=K.element_id and F.group_id=K.filter_group
    GROUP BY F.batch_id, F.group_id, K.filter_id, F.output_pos
),
Relu238_fwd0 AS (
    select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
        FROM Conv237_fwd0 F
),
Conv239_fwd0 AS (
    select batch_id, K.filter_id, F.tuple_id as tuple_id,
    sum(F.value*K.weight_value) as value
        FROM Relu238_fwd0 F INNER JOIN wt_conv239_weight K
        on F.kernel_id=K.element_id
    GROUP BY F.batch_id, K.filter_id, F.tuple_id
),
Add240_fwd0 AS (
    select A.batch_id, A.kernel_id, A.tuple_id,
    A.value + B.value as value
        FROM Relu234_fwd0 A INNER JOIN Conv239_fwd0 B
        on A.batch_id=B.batch_id and A.kernel_id=B.kernel_id
        and A.tuple_id=B.tuple_id
),
Relu241_fwd0 AS (
    select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
        FROM Add240_fwd0 F
),
Conv242_fwd0 AS (
    select batch_id, K.filter_id, F.tuple_id as tuple_id,
    sum(F.value*K.weight_value) as value
        FROM Relu241_fwd0 F INNER JOIN wt_conv242_weight K
        on F.kernel_id=K.element_id
    GROUP BY F.batch_id, K.filter_id, F.tuple_id
),
Relu241_fwd0_mapped AS (
    select batch_id, output_pos, kernel_pos, value
        FROM Relu241_fwd0 F INNER JOIN block2_downsample_map M
        on F.kernel_id = M.channel_id and F.tuple_id = M.input_pos
),
Conv243_fwd0 AS (
    select batch_id, K.filter_id, F.output_pos as tuple_id,
    sum(F.value*K.weight_value) as value
        FROM Relu241_fwd0_mapped F INNER JOIN wt_conv243_weight K
        on F.kernel_pos=K.element_id
    GROUP BY F.batch_id, K.filter_id, F.output_pos
),
Relu244_fwd0 AS (
    select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
        FROM Conv242_fwd0 F
),
Relu244_fwd0_mapped AS (
    select batch_id, group_id, output_pos, kernel_pos, value
        FROM Relu244_fwd0 F INNER JOIN block2_stage2_map M
        on F.kernel_id = M.channel_id and F.tuple_id = M.input_pos
),
Conv245_fwd0 AS (
    select batch_id, (F.group_id*8+filter_id) as kernel_id, F.output_pos as tuple_id,
    sum(F.value*K.weight_value) as value
        FROM Relu244_fwd0_mapped F INNER JOIN wt_conv245_weight K
        on F.kernel_pos=K.element_id and F.group_id=K.filter_group
    GROUP BY F.batch_id, F.group_id, K.filter_id, F.output_pos
),
Relu246_fwd0 AS (
    select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
        FROM Conv245_fwd0 F
),
Conv247_fwd0 AS (
    select batch_id, K.filter_id, F.tuple_id as tuple_id,
    sum(F.value*K.weight_value) as value
        FROM Relu246_fwd0 F INNER JOIN wt_conv247_weight K
        on F.kernel_id=K.element_id
    GROUP BY F.batch_id, K.filter_id, F.tuple_id
),
Add248_fwd0 AS (
    select A.batch_id, A.kernel_id, A.tuple_id,
    A.value + B.value as value
        FROM Conv243_fwd0 A INNER JOIN Conv247_fwd0 B
        on A.batch_id=B.batch_id and A.kernel_id=B.kernel_id
        and A.tuple_id=B.tuple_id
),
Relu249_fwd0 AS (
    select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
        FROM Add248_fwd0 F
),
Conv250_fwd0 AS (
    select batch_id, K.filter_id, F.tuple_id as tuple_id,
    sum(F.value*K.weight_value) as value
        FROM Relu249_fwd0 F INNER JOIN wt_conv250_weight K
        on F.kernel_id=K.element_id
    GROUP BY F.batch_id, K.filter_id, F.tuple_id
),
Relu251_fwd0 AS (
    select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
        FROM Conv250_fwd0 F
),
Relu251_fwd0_mapped AS (
    select batch_id, group_id, output_pos, kernel_pos, value
        FROM Relu251_fwd0 F INNER JOIN block2_stage3_map M
        on F.kernel_id = M.channel_id and F.tuple_id = M.input_pos
),
Conv252_fwd0 AS (
    select batch_id, (F.group_id*8+filter_id) as kernel_id, F.output_pos as tuple_id,
    sum(F.value*K.weight_value) as value
        FROM Relu251_fwd0_mapped F INNER JOIN wt_conv252_weight K
        on F.kernel_pos=K.element_id and F.group_id=K.filter_group
    GROUP BY F.batch_id, F.group_id, K.filter_id, F.output_pos
),
Relu253_fwd0 AS (
    select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
        FROM Conv252_fwd0 F
),
Conv254_fwd0 AS (
    select batch_id, K.filter_id, F.tuple_id as tuple_id,
    sum(F.value*K.weight_value) as value
        FROM Relu253_fwd0 F INNER JOIN wt_conv254_weight K
        on F.kernel_id=K.element_id
    GROUP BY F.batch_id, K.filter_id, F.tuple_id
),
Add255_fwd0 AS (
    select A.batch_id, A.kernel_id, A.tuple_id,
    A.value + B.value as value
        FROM Relu249_fwd0 A INNER JOIN Conv254_fwd0 B
        on A.batch_id=B.batch_id and A.kernel_id=B.kernel_id
        and A.tuple_id=B.tuple_id
),
Relu256_fwd0 AS (
    select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
        FROM Add255_fwd0 F
),
Conv257_fwd0 AS (
    select batch_id, K.filter_id, F.tuple_id as tuple_id,
    sum(F.value*K.weight_value) as value
        FROM Relu256_fwd0 F INNER JOIN wt_conv257_weight K
        on F.kernel_id=K.element_id
    GROUP BY F.batch_id, K.filter_id, F.tuple_id
),
Relu258_fwd0 AS (
    select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
        FROM Conv257_fwd0 F
),
Relu258_fwd0_mapped AS (
    select batch_id, group_id, output_pos, kernel_pos, value
        FROM Relu258_fwd0 F INNER JOIN block2_stage4_map M
        on F.kernel_id = M.channel_id and F.tuple_id = M.input_pos
),
Conv259_fwd0 AS (
    select batch_id, (F.group_id*8+filter_id) as kernel_id, F.output_pos as tuple_id,
    sum(F.value*K.weight_value) as value
        FROM Relu258_fwd0_mapped F INNER JOIN wt_conv259_weight K
        on F.kernel_pos=K.element_id and F.group_id=K.filter_group
    GROUP BY F.batch_id, F.group_id, K.filter_id, F.output_pos
),
Relu260_fwd0 AS (
    select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
        FROM Conv259_fwd0 F
),
Conv261_fwd0 AS (
    select batch_id, K.filter_id, F.tuple_id as tuple_id,
    sum(F.value*K.weight_value) as value
        FROM Relu260_fwd0 F INNER JOIN wt_conv261_weight K
        on F.kernel_id=K.element_id
    GROUP BY F.batch_id, K.filter_id, F.tuple_id
),
Add262_fwd0 AS (
    select A.batch_id, A.kernel_id, A.tuple_id,
    A.value + B.value as value
        FROM Relu256_fwd0 A INNER JOIN Conv261_fwd0 B
        on A.batch_id=B.batch_id and A.kernel_id=B.kernel_id
        and A.tuple_id=B.tuple_id
),
Relu263_fwd0 AS (
    select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
        FROM Add262_fwd0 F
),
Conv264_fwd0 AS (
    select batch_id, K.filter_id, F.tuple_id as tuple_id,
    sum(F.value*K.weight_value) as value
        FROM Relu263_fwd0 F INNER JOIN wt_conv264_weight K
        on F.kernel_id=K.element_id
    GROUP BY F.batch_id, K.filter_id, F.tuple_id
),
Relu265_fwd0 AS (
    select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
        FROM Conv264_fwd0 F
),
Relu265_fwd0_mapped AS (
    select batch_id, group_id, output_pos, kernel_pos, value
        FROM Relu265_fwd0 F INNER JOIN block3_stage1_map M
        on F.kernel_id = M.channel_id and F.tuple_id = M.input_pos
),
Conv266_fwd0 AS (
    select batch_id, (F.group_id*8+filter_id) as kernel_id, F.output_pos as tuple_id,
    sum(F.value*K.weight_value) as value
        FROM Relu265_fwd0_mapped F INNER JOIN wt_conv266_weight K
        on F.kernel_pos=K.element_id and F.group_id=K.filter_group
    GROUP BY F.batch_id, F.group_id, K.filter_id, F.output_pos
),
Relu267_fwd0 AS (
    select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
        FROM Conv266_fwd0 F
),
Conv268_fwd0 AS (
    select batch_id, K.filter_id, F.tuple_id as tuple_id,
    sum(F.value*K.weight_value) as value
        FROM Relu267_fwd0 F INNER JOIN wt_conv268_weight K
        on F.kernel_id=K.element_id
    GROUP BY F.batch_id, K.filter_id, F.tuple_id
),
Add269_fwd0 AS (
    select A.batch_id, A.kernel_id, A.tuple_id,
    A.value + B.value as value
        FROM Relu263_fwd0 A INNER JOIN Conv268_fwd0 B
        on A.batch_id=B.batch_id and A.kernel_id=B.kernel_id
        and A.tuple_id=B.tuple_id
),
Relu270_fwd0 AS (
    select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
        FROM Add269_fwd0 F
),
Conv271_fwd0 AS (
    select batch_id, K.filter_id, F.tuple_id as tuple_id,
    sum(F.value*K.weight_value) as value
        FROM Relu270_fwd0 F INNER JOIN wt_conv271_weight K
        on F.kernel_id=K.element_id
    GROUP BY F.batch_id, K.filter_id, F.tuple_id
),
Relu272_fwd0 AS (
    select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
        FROM Conv271_fwd0 F
),
Relu272_fwd0_mapped AS (
    select batch_id, group_id, output_pos, kernel_pos, value
        FROM Relu272_fwd0 F INNER JOIN block3_stage2_map M
        on F.kernel_id = M.channel_id and F.tuple_id = M.input_pos
),
Conv273_fwd0 AS (
    select batch_id, (F.group_id*8+filter_id) as kernel_id, F.output_pos as tuple_id,
    sum(F.value*K.weight_value) as value
        FROM Relu272_fwd0_mapped F INNER JOIN wt_conv273_weight K
        on F.kernel_pos=K.element_id and F.group_id=K.filter_group
    GROUP BY F.batch_id, F.group_id, K.filter_id, F.output_pos
),
Relu274_fwd0 AS (
    select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
        FROM Conv273_fwd0 F
),
Conv275_fwd0 AS (
    select batch_id, K.filter_id, F.tuple_id as tuple_id,
    sum(F.value*K.weight_value) as value
        FROM Relu274_fwd0 F INNER JOIN wt_conv275_weight K
        on F.kernel_id=K.element_id
    GROUP BY F.batch_id, K.filter_id, F.tuple_id
),
Add276_fwd0 AS (
    select A.batch_id, A.kernel_id, A.tuple_id,
    A.value + B.value as value
        FROM Relu270_fwd0 A INNER JOIN Conv275_fwd0 B
        on A.batch_id=B.batch_id and A.kernel_id=B.kernel_id
        and A.tuple_id=B.tuple_id
),
Relu277_fwd0 AS (
    select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
        FROM Add276_fwd0 F
),
Conv278_fwd0 AS (
    select batch_id, K.filter_id, F.tuple_id as tuple_id,
    sum(F.value*K.weight_value) as value
        FROM Relu277_fwd0 F INNER JOIN wt_conv278_weight K
        on F.kernel_id=K.element_id
    GROUP BY F.batch_id, K.filter_id, F.tuple_id
),
Relu277_fwd0_mapped AS (
    select batch_id, output_pos, kernel_pos, value
        FROM Relu277_fwd0 F INNER JOIN block3_downsample_map M
        on F.kernel_id = M.channel_id and F.tuple_id = M.input_pos
),
Conv279_fwd0 AS (
    select batch_id, K.filter_id, F.output_pos as tuple_id,
    sum(F.value*K.weight_value) as value
        FROM Relu277_fwd0_mapped F INNER JOIN wt_conv279_weight K
        on F.kernel_pos=K.element_id
    GROUP BY F.batch_id, K.filter_id, F.output_pos
),
Relu280_fwd0 AS (
    select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
        FROM Conv278_fwd0 F
),
Relu280_fwd0_mapped AS (
    select batch_id, group_id, output_pos, kernel_pos, value
        FROM Relu280_fwd0 F INNER JOIN block3_stage3_map M
        on F.kernel_id = M.channel_id and F.tuple_id = M.input_pos
),
Conv281_fwd0 AS (
    select batch_id, (F.group_id*16+filter_id) as kernel_id, F.output_pos as tuple_id,
    sum(F.value*K.weight_value) as value
        FROM Relu280_fwd0_mapped F INNER JOIN wt_conv281_weight K
        on F.kernel_pos=K.element_id and F.group_id=K.filter_group
    GROUP BY F.batch_id, F.group_id, K.filter_id, F.output_pos
),
Relu282_fwd0 AS (
    select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
        FROM Conv281_fwd0 F
),
Conv283_fwd0 AS (
    select batch_id, K.filter_id, F.tuple_id as tuple_id,
    sum(F.value*K.weight_value) as value
        FROM Relu282_fwd0 F INNER JOIN wt_conv283_weight K
        on F.kernel_id=K.element_id
    GROUP BY F.batch_id, K.filter_id, F.tuple_id
),
Add284_fwd0 AS (
    select A.batch_id, A.kernel_id, A.tuple_id,
    A.value + B.value as value
        FROM Conv279_fwd0 A INNER JOIN Conv283_fwd0 B
        on A.batch_id=B.batch_id and A.kernel_id=B.kernel_id
        and A.tuple_id=B.tuple_id
),
Relu285_fwd0 AS (
    select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
        FROM Add284_fwd0 F
),
Conv286_fwd0 AS (
    select batch_id, K.filter_id, F.tuple_id as tuple_id,
    sum(F.value*K.weight_value) as value
        FROM Relu285_fwd0 F INNER JOIN wt_conv286_weight K
        on F.kernel_id=K.element_id
    GROUP BY F.batch_id, K.filter_id, F.tuple_id
),
Relu287_fwd0 AS (
    select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
        FROM Conv286_fwd0 F
),
Relu287_fwd0_mapped AS (
    select batch_id, group_id, output_pos, kernel_pos, value
        FROM Relu287_fwd0 F INNER JOIN block3_stage4_map M
        on F.kernel_id = M.channel_id and F.tuple_id = M.input_pos
),
Conv288_fwd0 AS (
    select batch_id, (F.group_id*16+filter_id) as kernel_id, F.output_pos as tuple_id,
    sum(F.value*K.weight_value) as value
        FROM Relu287_fwd0_mapped F INNER JOIN wt_conv288_weight K
        on F.kernel_pos=K.element_id and F.group_id=K.filter_group
    GROUP BY F.batch_id, F.group_id, K.filter_id, F.output_pos
),
Relu289_fwd0 AS (
    select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
        FROM Conv288_fwd0 F
),
Conv290_fwd0 AS (
    select batch_id, K.filter_id, F.tuple_id as tuple_id,
    sum(F.value*K.weight_value) as value
        FROM Relu289_fwd0 F INNER JOIN wt_conv290_weight K
        on F.kernel_id=K.element_id
    GROUP BY F.batch_id, K.filter_id, F.tuple_id
),
Add291_fwd0 AS (
    select A.batch_id, A.kernel_id, A.tuple_id,
    A.value + B.value as value
        FROM Relu285_fwd0 A INNER JOIN Conv290_fwd0 B
        on A.batch_id=B.batch_id and A.kernel_id=B.kernel_id
        and A.tuple_id=B.tuple_id
),
Relu292_fwd0 AS (
    select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
        FROM Add291_fwd0 F
),
Conv293_fwd0 AS (
    select batch_id, K.filter_id, F.tuple_id as tuple_id,
    sum(F.value*K.weight_value) as value
        FROM Relu292_fwd0 F INNER JOIN wt_conv293_weight K
        on F.kernel_id=K.element_id
    GROUP BY F.batch_id, K.filter_id, F.tuple_id
),
Relu294_fwd0 AS (
    select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
        FROM Conv293_fwd0 F
),
Relu294_fwd0_mapped AS (
    select batch_id, group_id, output_pos, kernel_pos, value
        FROM Relu294_fwd0 F INNER JOIN block4_stage1_map M
        on F.kernel_id = M.channel_id and F.tuple_id = M.input_pos
),
Conv295_fwd0 AS (
    select batch_id, (F.group_id*16+filter_id) as kernel_id, F.output_pos as tuple_id,
    sum(F.value*K.weight_value) as value
        FROM Relu294_fwd0_mapped F INNER JOIN wt_conv295_weight K
        on F.kernel_pos=K.element_id and F.group_id=K.filter_group
    GROUP BY F.batch_id, F.group_id, K.filter_id, F.output_pos
),
Relu296_fwd0 AS (
    select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
        FROM Conv295_fwd0 F
),
Conv297_fwd0 AS (
    select batch_id, K.filter_id, F.tuple_id as tuple_id,
    sum(F.value*K.weight_value) as value
        FROM Relu296_fwd0 F INNER JOIN wt_conv297_weight K
        on F.kernel_id=K.element_id
    GROUP BY F.batch_id, K.filter_id, F.tuple_id
),
Add298_fwd0 AS (
    select A.batch_id, A.kernel_id, A.tuple_id,
    A.value + B.value as value
        FROM Relu292_fwd0 A INNER JOIN Conv297_fwd0 B
        on A.batch_id=B.batch_id and A.kernel_id=B.kernel_id
        and A.tuple_id=B.tuple_id
),
Relu299_fwd0 AS (
    select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
        FROM Add298_fwd0 F
),
Conv300_fwd0 AS (
    select batch_id, K.filter_id, F.tuple_id as tuple_id,
    sum(F.value*K.weight_value) as value
        FROM Relu299_fwd0 F INNER JOIN wt_conv300_weight K
        on F.kernel_id=K.element_id
    GROUP BY F.batch_id, K.filter_id, F.tuple_id
),
Relu301_fwd0 AS (
    select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
        FROM Conv300_fwd0 F
),
Relu301_fwd0_mapped AS (
    select batch_id, group_id, output_pos, kernel_pos, value
        FROM Relu301_fwd0 F INNER JOIN block4_stage2_map M
        on F.kernel_id = M.channel_id and F.tuple_id = M.input_pos
),
Conv302_fwd0 AS (
    select batch_id, (F.group_id*16+filter_id) as kernel_id, F.output_pos as tuple_id,
    sum(F.value*K.weight_value) as value
        FROM Relu301_fwd0_mapped F INNER JOIN wt_conv302_weight K
        on F.kernel_pos=K.element_id and F.group_id=K.filter_group
    GROUP BY F.batch_id, F.group_id, K.filter_id, F.output_pos
),
Relu303_fwd0 AS (
    select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
        FROM Conv302_fwd0 F
),
Conv304_fwd0 AS (
    select batch_id, K.filter_id, F.tuple_id as tuple_id,
    sum(F.value*K.weight_value) as value
        FROM Relu303_fwd0 F INNER JOIN wt_conv304_weight K
        on F.kernel_id=K.element_id
    GROUP BY F.batch_id, K.filter_id, F.tuple_id
),
Add305_fwd0 AS (
    select A.batch_id, A.kernel_id, A.tuple_id,
    A.value + B.value as value
        FROM Relu299_fwd0 A INNER JOIN Conv304_fwd0 B
        on A.batch_id=B.batch_id and A.kernel_id=B.kernel_id
        and A.tuple_id=B.tuple_id
),
Relu306_fwd0 AS (
    select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
        FROM Add305_fwd0 F
),
Conv307_fwd0 AS (
    select batch_id, K.filter_id, F.tuple_id as tuple_id,
    sum(F.value*K.weight_value) as value
        FROM Relu306_fwd0 F INNER JOIN wt_conv307_weight K
        on F.kernel_id=K.element_id
    GROUP BY F.batch_id, K.filter_id, F.tuple_id
),
Relu308_fwd0 AS (
    select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
        FROM Conv307_fwd0 F
),
Relu308_fwd0_mapped AS (
    select batch_id, group_id, output_pos, kernel_pos, value
        FROM Relu308_fwd0 F INNER JOIN block4_stage3_map M
        on F.kernel_id = M.channel_id and F.tuple_id = M.input_pos
),
Conv309_fwd0 AS (
    select batch_id, (F.group_id*16+filter_id) as kernel_id, F.output_pos as tuple_id,
    sum(F.value*K.weight_value) as value
        FROM Relu308_fwd0_mapped F INNER JOIN wt_conv309_weight K
        on F.kernel_pos=K.element_id and F.group_id=K.filter_group
    GROUP BY F.batch_id, F.group_id, K.filter_id, F.output_pos
),
Relu310_fwd0 AS (
    select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
        FROM Conv309_fwd0 F
),
Conv311_fwd0 AS (
    select batch_id, K.filter_id, F.tuple_id as tuple_id,
    sum(F.value*K.weight_value) as value
        FROM Relu310_fwd0 F INNER JOIN wt_conv311_weight K
        on F.kernel_id=K.element_id
    GROUP BY F.batch_id, K.filter_id, F.tuple_id
),
Add312_fwd0 AS (
    select A.batch_id, A.kernel_id, A.tuple_id,
    A.value + B.value as value
        FROM Relu306_fwd0 A INNER JOIN Conv311_fwd0 B
        on A.batch_id=B.batch_id and A.kernel_id=B.kernel_id
        and A.tuple_id=B.tuple_id
),
Relu313_fwd0 AS (
    select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
        FROM Add312_fwd0 F
),
AveragePool315_fwd0 AS (
    select batch_id, kernel_id, avg(value) as value
        FROM Relu313_fwd0
    GROUP BY batch_id, kernel_id
),
MatMul318_fwd0 AS (
    select batch_id, K.output_unit,
    sum(F.value*K.weight_value) as value
        FROM AveragePool315_fwd0 F INNER JOIN wt_fc_weight K
        on F.kernel_id=K.input_unit
    GROUP BY F.batch_id, K.output_unit
),
SELECT l.name AS res FROM
cifar10_labels l
JOIN (
    SELECT DISTINCT on (batch_id) batch_id, kernel_id+1 AS label
    FROM MatMul318_fwd0 ORDER BY batch_id, value DESC) t
    ON t.label = l.label
